# Домашнее задание 1 (часть 1)

Всего 2 части, по 50 баллов.

Ваши ФАМИЛИЯ, ИМЯ, ГРУППА

Для проверяющего:

Оценка: ?? из 50

In [ ]:
#пожалуйста, все импорты здесь

#comments in english since too much time for me to switch
import numpy as np
import matplotlib.pyplot as plt
import pickle

import tqdm #better to make extensive use of it!

from PIL import Image #visuals

## Отпуск (20 баллов)

**(1)** Андрей Алексеевич (АА, имя изменено) уехал в отпуск на $T$ дней (надеемся, что $T>0$). У него есть стартовый капитал $x_0>0$, а отпуск он отмечает так, что получает от использования $u$ денег полезность $f(u)=u^{1-\gamma}, ~\gamma \in (0,1)$. Его цель -- максимизировать эффект от своего отпуска и от того, сколько у него останется в конце:

$$
\max_{(u_t)} \sum_{t=0}^{T-1} \beta^t u_t^{1-\gamma} + \beta^T A x_T^{1-\gamma},
$$

где $A>0, \gamma \in (0,1)$ -- это заданные константы. Если АА решает тратить $u_t$ денег в момент $t$, то его бюджет $x_t$ изменяется на

$$
x_{t+1} = a_t (x_t - u_t),
$$

при этом $u_t \in [0,x_t]$ и ещё $x_t \geq 0$, так как занимать в долг он не готов. Последовательность $a_t$ считается заданной детерминированной последовательностью, которая позволяет учесть изменяющий курс валюты (АА использует отечественную банковскую карту).

Проверьте, что задача имеет решение и найдите оптимальное управление $u$ с помощью принципа Беллмана.

**(2) (+5 баллов, бонус)** Пока АА находился в отпуске, в экономике начали происходить потрясения, из-за которых доля изменения его бюджета $ a_t $ стала случайной $ Z_t $. Полагая, что $ {Z}_{t=0}^{T-1} $ - последовательность положительных независимых одинаково распределенных случайных величин с плотностью $ p(z_t) $ выпишите уравнения Беллмана. Как решение задачи будет отличаться от детерминированного случая?

ЗДЕСЬ МОГЛО БЫТЬ ВАШЕ РЕШЕНИЕ

## Ищем клад (30 баллов)

Очень известный пират капитан Дренсис Фейк (ДФ, самый известный после Николаса Шарпа) получил в свои руки карту острова с расположением нескольких кладов и отправился их откапывать. Про каждый клад известна его ценность $V_k, ~k=1,..,K$.

Корабль попал в шторм, но устоял, хотя оказался в итоге с той стороны острова, где не планировали. Пока команда занимается ремонтом, Дренсису нужно успеть откопать клад и отнести его обратно, потратив минимальные усилия и доставив самый солидный приз. В этой задаче есть много нюансов, но их можно разрешить, если, как рекомендовал Ричард Беллман, спокойно о них подумать.

## Постановка задачи

1. Каждый свой ход ДФ устаёт в зависимости от окружающего ландшафта, но он достаточно выносливый и ради клада готов хорошо напрячься, поэтому сил у него без ограничений.
2. Каждый клад ценится по-разному и принести на корабль можно только один.
3. Когда ДФ возвращается с кладом на плечах, его усилия расходуются в 2 раза интенсивнее (Пример: в обычных услоиях в момент $t$ тратится 5, с кладом будет 10).
4. ДФ максимизирует следующий функционал:
   
   $$
        -\sum_{t=0}^{T-1} f(t,X_t,u_t) + S(X_T),
   $$

где $X_t \in \mathbb{Z}_{\geq 0}^2$ показывает состояние (координаты), $u_t \in \lbrace 1,..,8\rbrace$ направление движения, функция $f$ показывает сколько усилий будет затрачено на переход. Терминант $S$ удобно использовать для того, чтобы учесть ценность принесённого клада. 

Один интересный нюанс состоит в том, что $T$ не фиксировано, потому что разное число ходов может потребоваться для того, чтобы дойти до каждого из кладов.

## Конкретные параметры

Начнём с того, что у нас есть карта, давайте на неё посмотрим.

In [ ]:
with open("map.map","rb") as f:
    dic = pickle.load(f)
    theMap = dic["map"]

In [ ]:
#The Map
# inspired by https://github.com/wadefletch/terrain
class Color:
    DEEP_OCEAN = (94, 129, 172)
    OCEAN = (115, 146, 183)
    BEACH = (235, 203, 139)
    LAND = (143, 176, 115)
    FOREST = (163, 190, 140)
    MOUNTAIN = (76, 86, 106)
    HIGH_MOUNTAIN = (121, 133, 159)
    SNOW = (216, 222, 233)
    CITY = (255, 0, 0)

class Map:
    
    def __init__(self, map):
        self.map = map
        #parameters
        
    def to_image(self, filename=None, grayscale=False):
        def color(e):
            if e >= 0.85:
                return Color.SNOW
            elif e >= 0.75:
                return Color.HIGH_MOUNTAIN
            elif e >= 0.65:
                return Color.MOUNTAIN
            elif e >= 0.50:
                return Color.FOREST
            elif e >= 0.37:
                return Color.LAND
            elif e >= 0.35:
                return Color.BEACH
            elif e >= 0.20:
                return Color.OCEAN
            else:
                return Color.DEEP_OCEAN

        if grayscale:
            self._normalize()  # TODO: maybe remove?
            grayscale_world = self.map * 255.0
            im = Image.fromarray(np.uint8(grayscale_world), "L").save(filename)
            return

        image_data = [
            color(self.map[y][x])
            for x in range(self.map.shape[0])
            for y in range(self.map.shape[1])
        ]

        im = Image.new("RGB", self.map.shape,"white")#self.map.shape[::-1], "white")
        im.putdata(image_data)
        if( not (filename is None) ):
            im.save(filename)
        return im
        

    def draw(self, treasures=None, start=None):
        """Draws the map with treasures (if given)

        Args:
            treasures (list(dict), optional): Treasures dict of pairs {"location": (x,y), "value": float}. Defaults to None.
            start (int[], optional): start location (x,y) (2,). Defaults to None.

        Returns:
            ax: axes
        """             
        f, ax = plt.subplots(figsize=(7,7))

        ax.set_title("Карта")
        ax.imshow(self.to_image())
        
        if(treasures is not None):
            for tr in treasures:
                ax.scatter(tr["location"][0],tr["location"][1],200,marker="x",color="red")
            
        if(start is not None):
            ax.scatter(start[0],start[1],20,marker="o",color="green")
        return ax
    

In [ ]:
#load!
with open("map.map","rb") as f:
    dd= pickle.load(f)

mapObj = Map(...???)

ax=mapObj.draw(treasures, start)
plt.show()

Наша задача похожа на задачу поиска пути, но с некоторыми нюансами, которые требуют размышлений.

1. Затраченная энергия зависит от перепада высот. Математически это выражается так:
   
   $$
   f(X_t,u_t) = 90000(  m(X_{t+1}) - m(X_t))^2,
   $$

   Число $90000$ -- это поправочный коэффициент на масштаб карты высот. Не забываем, что в случае если идём с кладом, $f$ дополнительно умножается на $2$.

2. Кажется, что это классическая задача поиска пути, это почти так, но есть терминант. Тем не менее, решать мы её будем методом динамического програмиирования, который на самом деле будет легко модифицированным алгоритмом Дейкстры. Матрица расстояний, получаемая в результате алгоритма Дейкстры (если искать путь из конца в начало), и есть та самая функция Беллмана.
   
3. Время $T$ не фиксировано, потому что заранее неясно, сколько мы будем туда идти.

Это технические моменты, которые будем понемногу решать, связывая новую математику со старыми добрыми алгоритмами поисками пути.

## Метод решения

1. Решаем задачу поиска оптимальной траектории для каждого выбора клада.
2. Выбираем лучший клад (и соответсвенно лучшую траекторию) с точки зрения целевой функции.

Сформулируем задачу.

**Целевая функция**

$$
-\sum_{t=0}^{T-1} f(X_t,u_t) + S(X_T)
$$

нужно её максимизировать по $u_0,..,u_{T-1},X_1,..,X_T$ и (!!) по $T$. Переменная $X_t$ -- это состояние ДФ в момент $t$, о нём ниже.

Мы решили, что идём к конкретному кладу с ценностью $v$, поэтому $S(X_T)=v$ только если в момент $T$ координата совпадает с координатой клада, иначе 0.

**Динамика**

Состояние обозначим за $X_t = [x_t,y_t]^T$, соответственно, координаты. После одного шага оно изменяется на $x_{t+1},y_{t+1}$ в зависимости от выбранного направления, при этом у карты есть границы, за них нельзя выходить.

**Уравнение Беллмана**

Положим пока $T$ заданным и

$$
J_t(X_t) = \max_{u_t,u_{t+1},...,u_{T-1},T}\left[ -\sum_{j=t}^{T-1} f(X_j,u_j) + S(X_T) \right]
$$

для $t<T$ и для $t=T$

$$
J_T(X_T) = S(X_T).
$$

Докажите принцип динамического программирования для $J_t(X_t)$ при $t<T$:

$$
J_t(X_t) = \max_{u_t}\left[ -f(X_t,u_t) + J_{t+1}(X_{t+1}) \right].
$$

ВАШЕ ДОКАЗАТЕЛЬСТВО ЗДЕСЬ

## Имплементация

Мы построили строгий математический алгоритм, но пока никак не учли, что $T$ неизвестно заранее. К счастью для нас, любая оптимальная траектория должна быть конечна (то есть, $T$ конечно) и должна приходить к одному из кладов. Почему?

ВАШЕ ОБОСНОВАНИЕ ТУТ

Следовательно, мы можем запустить алгоритм обратной индукции и решить уравнение Беллмана. Проблема в том, что мы знаем, где индукция начинается (в точке клада), но не знаем, чему равно $T$.

Но вообще, мы, конечно, можем смело считать время от конца, сосредоточившись на поиске лучшего пути, используя

$$
J_t(X_t) = \max_{u_t}\left[ -f(X_t,u_t) + J_{t+1}(X_{t+1}) \right],
$$

а потом уже определить $T$, когда мы дойдём до стартовой точки.

Для "всех $X_t$" автоматически означает для всех клеток $(x,y)$. Поскольку шаги идут от конца, множество достижимых клеток относительно небольшое: на шаге $T-t$ оно будет составлять всех соседей в пределах $t$ ходов. Функцию Беллмана $J_{T-t}$ будем вычислять только для достижимых вершин и доопределять какой-то страшной отрицательной константой в остальной области.

Оказывается, что для всех $t$ функция Беллмана $J_t(X)=J(X)$, то есть, она не зависит от времени. Это нужно доказать.

ВАШЕ ДОКАЗАТЕЛЬСТВО ЗДЕСЬ

Для нас это большой плюс: вместо построения последовательности функций Беллмана, нужно всего лишь решить рекурсивное уравнение

$$
J(X) = \max_{u_t}\left[ -f(X,u) + J( X' ) \right], 
$$

где $X'$ получается переходом из $X$ с помощью применённого управления $u$. Иными словами, правильно делая обратную индукцию от финиша, понемногу заполнять таблицу $J(X)$ наибольшими выгодами от начала пути из заданной клетки.

## Когда же код?

### Функция Беллмана

Мы дошли и вполне готовы написать код. На самом деле даже вычисление одного шага назад может показаться нетрививальным. Представьте ситуацию, когда клад за горой и за один переход дёшево до него не дойти. К счастью для нас есть схема, которая позволяет учесть и это.

Идея в следующем.

Изначально формируется множество непосещённых клеток, только в одной мы знаем функцию Беллмана -- это в точке клада.

Мы начинаем с $x_T$, точки с кладом, в ней функция Беллмана равна ценности клада. Шаг обратной индукции состоит в исследовании соседей, но тут есть сложность, потому что у соседей тоже есть соседи и самый выгодный путь до $x_T$ не обязательно будет в один шаг. Этот момент учитывается очень креативно.

Пока мы находимся в  $x_T$, у всех соседей мы предварительно отмечаем длину пути до этой вершины (изначально в непосещённых вершинах стоит большое отрицательное число). Далее, мы отмечаем $x_T$ как посещённую клетку (мы оценили цены вокруг и вычислили функцию Беллмана) и мы делаем переход в соседа, откуда идёт наивыгоднейший путь, и вычисляем функцию Беллмана для него. Гарантируется, что этот путь из этого соседа до $x_T$ для него будет самым выгодным, потому что в обход мы начинаем терять дополнительные силы на переход в соседа и потом из соседа в $x_T$. 

Далее мы смотрим на список непосещённых клеток, где уже есть какая-то оценка пути, выбираем клетку с самым выгодным значением для исследования и повторяем операцию выше, обновляя веса вокруг и помечая вершину как посещённую.

Всё это на самом деле алгоритм Дейкстры, если мы забудем про существование терминанта и заменим максимум на минимум.

Преимущество в обладании функцией Беллмана на таком графе состоит в том, что из любой точки можно легко построить оптимальный маршрут за какое-то конечное время (выбранное оптимально по конструкции).

Запишите псевдокод алгоритма с учётом всех обсуждённых нюансов.

ВАШ ПСЕВДОКОД

Интересно, что Дейкстра и другие классики построили этот алгоритм примерно в то время, когда Беллман придумал динамическое программирование. Связано ли это с хорошим нетворкингом или просто совпадением, сказать сложно, но принцип выбора вершины с самым коротким путём -- это принцип динамического программирования, который верен в задаче поиска пути. Но при этом обратная индукция устроена несколько сложнее и функция Беллмана вычисляется как бы на всех слоях по времени сразу, постепенно исследуя пространство и пользуясь тем, что функция Беллмана не зависит от времени.

### Поиск пути

Функция Беллмана содержит всю информацию об оптимальных путях и их выгодах. Опишите, как по вычисленной функции Беллмана найти оптимальный путь.

ВАШ МЕТОД ЗДЕСЬ

## Пишем решение

Вопреки традиции пишем аккуратно, так как нам ещё проводить эксперименты.

Обратите внимание, что ДФ не может плыть по морю, он ходит по суше, поэтому океан и озёра считается непроходимыми, их легко найти по карте, согласно цветам выше высота ``<0.35`` считается водой. Но на гору он готов залезть, если потребуется.

Рекурсии стоит избегать, потому что карта в теории может быть очень неприятных масштабов.

In [ ]:
#comments in English to use autodocstring
#feel free to add your own methods if necessary
class TreasureHunter:
    
    def __init__(self,themap,treasures):
        self.map = themap # (N,M)
        self.values = None # can be (1,N,M) or (K,N,M) if there are K treasures
        self.treasures = treasures
        self.bigConst = 100500100500
        
    def _preprocessValues(self):
        """
        Does the initial set up of Bellman function

        1) sets -inf cost for all ocean cells
        """        
        self.values = -self.bigConst*np.ones([len(self.treasures),
                                              self.map.map.shape[0],
                                              self.map.map.shape[1]])
        
    def buildValues(self):
        """
        Computes Bellman function for all nodes
        """        
        raise NotImplementedError                      
    
    def computePath(self,x):
        """Computes optimal path from point x to the treasure(s)

        Args:
            x (float[]): start point (2,)

        Returns:
            float, int, list[int[]]: prize, treasureId, trajectory (list of points (2,) )
        """                
        raise NotImplementedError

    def _printTreasures(self):
        #printing stuff
        st = ""
        for tr in self.treasures:
            st = st + str(tr)+"\n      "
        return st
        
    def __str__(self):
        #printing stuff
        st = "TreasureHunter, \n    "
        st = st + f"map: {self.map.map.shape}\n    "
        st = st + f"treasures: \n      {self._printTreasures()}"
        return st

Тестируем сначала на небольшой задаче.

In [ ]:
mapTest = Map(np.random.uniform(size=(5,6)))
treasures = [{"location":np.array([2,3]), "value": 8000},{"location":np.array([3,5]), "value": 9000}]
hunter = TreasureHunter(mapTest,treasures)
print(hunter)

In [ ]:
hunter.buildValues()

In [ ]:
print(hunter.values)

In [ ]:
print(hunter.map.map)

In [ ]:
start = np.array([0,0])
hunter.computePath(start)

Убедитесь, что пути найдены верно и примерно с правильными издержками.

## Проверяем на данной задаче

Теперь воспользуемся нашей картой.

(очень красивая, нарисуем ещё раз)

In [ ]:
ax=mapObj.draw(treasures, start)
plt.show()

Примените ваш алгоритм, чтобы найти и отобразить на карте оптимальный путь. 

(Внимание, возможно, вам придётся подождать неторопливый ``Python`` минут 20..40, можете заварить кофе)

In [ ]:
DFHunter = TreasureHunter(themap=mapObj, treasures=treasures)

In [ ]:
DFHunter.buildValues()

УРА

Сохраним, пригодится, если дальше ноутбук сломается.

In [ ]:
with open("./resReal.hunter","wb") as f:
    pickle.dump({"hunter":DFHunter},f)

Напечатайте три функции Беллмана, используя ``matplotlib.pyplot.imshow``. 

Вам в помощь даётся код внизу, который рисует карту в правильной ориентации.

In [ ]:
def plotBellmanFun(vals, tit, treasure=None):
    f, ax = plt.subplots(figsize=(7,7))

    ax.set_title(tit,fontsize=16)
    mmap = (vals==-100500100500)*(np.amin(vals[(vals>-100500100500)])) + vals*(vals>-100500100500)
    im=ax.imshow(mmap.T, cmap="hot")
    plt.colorbar(im)
    if(treasure is not None):
        ax.scatter(treasure["location"][0],treasure["location"][1],marker="x",s=50,color="red")
    return ax
???


Найдите самый выгодный путь из старта и напечатайте его вместе с соответствующей кладу функцией Беллмана.

In [ ]:
_,_,points = DFHunter.computePath(start)
points = np.concatenate([point[None,:] for point in points],axis=0)

#plots...

## Финал и геометрия кладов

Мы почти решили задачу, остаётся только одна небольшая деталь: учесть обратный путь (который 2х тяжелее по сравнению с путём к кладу). Опишите способ, как учесть это в вашем решении и выведите значение целевой функции в точке старта при условии оптимального управления.

ВАШИ МЫСЛИ ЗДЕСЬ

In [ ]:
#ВАШ ОТВЕТ ТУТ
print(treasureId?, value? )

Учитывая такие необычные пути (жизнь в этом смысле непростая), как по точке понять самый стартовый клад? На это тоже есть способ. Нарисуйте карту острова и покрасьте её прозрачной заливкой цветами, соответствующими выбираемому кладу в данной точке.

In [ ]:
??

In [ ]:
??

## Где самая выгодная локация для высадки?

На этот вопрос тожно можно ответить с помощью посчитанной функции Беллмана. Как?

ВАШИ ОБОСНОВАНИЯ ТУТ

Покажите на карте по аналогии с тем, как вы рисовали выше функции Беллмана для отдельных кладов.

In [ ]:
??

Ну бывает, что не везёт с погодой.

Но зима в начале 2026 удаётся на славу :)